
# Commercial Intelligence & Profitability Analytics

## 04 — Concentración comercial y rentabilidad bruta estimada

**Empresa:** Distribuidora Cordillera (datos ficticios)  
**Tecnologías:** SQLite, SQL avanzado, Jupyter Notebook  
**Fuente:** `cordillera_analytics.sqlite`

### Objetivo general

Evaluar la concentración del importe neto asociado a pedidos entregados y la distribución de los márgenes brutos estimados entre clientes, productos y categorías comerciales.

El análisis busca identificar patrones de dependencia comercial, diferencias de rentabilidad y oportunidades de investigación que permitan fundamentar decisiones de negocio.

### Alcance

1. Concentración comercial y clasificación ABC de clientes.
2. Concentración de productos y categorías.
3. Evaluación de márgenes brutos estimados.
4. Síntesis de hallazgos, riesgos y oportunidades comerciales.

### Metodología

Se utilizará el modelo dimensional desarrollado y validado en el Notebook 02, considerando los resultados descriptivos obtenidos en el Notebook 03.

Se aplicarán agregaciones SQL, expresiones comunes de tabla (CTE), funciones de ventana, participaciones relativas y análisis de concentración.

### Criterios de interpretación

Los indicadores monetarios considerarán exclusivamente pedidos con estado `entregado`, salvo indicación expresa.

El importe neto entregado no equivale necesariamente a facturación emitida ni a ingresos efectivamente cobrados.

Los márgenes brutos representan estimaciones basadas en los valores registrados en las líneas de pedido y no equivalen a utilidad neta.

Las clasificaciones ABC son herramientas descriptivas. No implican automáticamente decisiones de eliminación, retención o priorización comercial.


In [163]:
%load_ext sql
%sql sqlite:///../data/cordillera_analytics.sqlite

The sql extension is already loaded. To reload it, use:
  %reload_ext sql



## 4.1 Concentración comercial de clientes

### 4.1.1 Participación y acumulación del importe neto

#### Objetivo

Evaluar la distribución del importe neto de pedidos entregados entre los clientes registrados, identificando su contribución individual y acumulada al resultado comercial.

#### Metodología

Se construirá un resumen de importes netos a nivel de cliente utilizando `dim_cliente` y `fact_detalle_pedidos`.

Los clientes se ordenarán de mayor a menor importe neto entregado.

Posteriormente se calcularán dos indicadores:

- Participación individual sobre el importe neto total.
- Participación acumulada siguiendo el orden descendente de contribución monetaria.

Se conservarán los clientes sin pedidos entregados para mantener la totalidad de la cartera.

#### Consideraciones

La concentración se evaluará sobre importes netos registrados en pedidos entregados, no sobre ingresos cobrados ni utilidad neta.

La participación acumulada dependerá del ordenamiento de los clientes.

Los resultados permitirán evaluar concentración comercial, sin asumir previamente una distribución Pareto determinada.


In [164]:
%%sql

WITH resumen_clientes AS (
SELECT
    c.cliente_id,
    c.razon_social,
    c.segmento,
    COALESCE(SUM(t.importe_neto_linea_clp), 0) AS importe_neto_entregado_clp
FROM dim_cliente AS c
LEFT JOIN fact_detalle_pedidos AS t
    ON c.cliente_id = t.cliente_id
    AND t.estado = 'entregado'
GROUP BY c.cliente_id, c.razon_social, c.segmento
)
SELECT
  *,
  ROUND(100.0 * importe_neto_entregado_clp /
   SUM(importe_neto_entregado_clp) OVER(), 2) AS participacion_pct,
ROUND(
    100.0 * SUM(importe_neto_entregado_clp) OVER(
    ORDER BY importe_neto_entregado_clp DESC, cliente_id ASC
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
) / SUM(importe_neto_entregado_clp) OVER(), 2) AS participacion_acumulada_pct
FROM resumen_clientes
ORDER BY importe_neto_entregado_clp DESC, cliente_id ASC;



Running query in 'sqlite:///../data/cordillera_analytics.sqlite'

cliente_id,razon_social,segmento,importe_neto_entregado_clp,participacion_pct,participacion_acumulada_pct
52,Almacen Ficticio Cordillera 0052,Almacen,48125814,1.09,1.09
67,Almacen Ficticio Cordillera 0067,Almacen,45871116,1.03,2.12
57,Almacen Ficticio Cordillera 0057,Almacen,45676530,1.03,3.15
48,Minimarket Ficticio Cordillera 0048,Minimarket,45091854,1.02,4.17
54,Minimarket Ficticio Cordillera 0054,Minimarket,43859664,0.99,5.16
63,Almacen Ficticio Cordillera 0063,Almacen,43667496,0.98,6.14
74,Almacen Ficticio Cordillera 0074,Almacen,43276812,0.98,7.12
73,Almacen Ficticio Cordillera 0073,Almacen,42543000,0.96,8.08
72,Almacen Ficticio Cordillera 0072,Almacen,42428298,0.96,9.03
64,Almacen Ficticio Cordillera 0064,Almacen,42312942,0.95,9.99


In [165]:
%%sql

WITH resumen_clientes AS (
SELECT
    c.cliente_id,
    c.razon_social,
    c.segmento,
    COALESCE(SUM(t.importe_neto_linea_clp), 0) AS importe_neto_entregado_clp
FROM dim_cliente AS c
LEFT JOIN fact_detalle_pedidos AS t
    ON c.cliente_id = t.cliente_id
    AND t.estado = 'entregado'
GROUP BY c.cliente_id, c.razon_social, c.segmento
), concentracion_clientes AS (
SELECT
  *,
  ROW_NUMBER() OVER ( 
    ORDER BY importe_neto_entregado_clp DESC, cliente_id ASC
        ) AS posicion_cliente,
  ROUND(100.0 * importe_neto_entregado_clp /
   SUM(importe_neto_entregado_clp) OVER(), 2) AS participacion_pct,
    100.0 * SUM(importe_neto_entregado_clp) OVER(
    ORDER BY importe_neto_entregado_clp DESC, cliente_id ASC
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
) / SUM(importe_neto_entregado_clp) OVER() AS participacion_acumulada_pct
FROM resumen_clientes
)
SELECT
    MIN(posicion_cliente) AS clientes_para_80_pct,
    ROUND(
    100.0 * MIN(posicion_cliente) / (SELECT COUNT(*) FROM resumen_clientes)
    ,2) AS pct_clientes_cartera,
    ROUND(
        MIN(participacion_acumulada_pct),2
    ) AS pct_importe_alcanzado
FROM concentracion_clientes
WHERE participacion_acumulada_pct >= 80
;




Running query in 'sqlite:///../data/cordillera_analytics.sqlite'

clientes_para_80_pct,pct_clientes_cartera,pct_importe_alcanzado
212,35.33,80.03



### 4.1.2 Concentración del 80 % del importe neto

#### Resultados

El análisis de concentración identificó que 212 de los 600 clientes registrados, equivalentes al 35,33 % de la cartera, acumulan el 80,03 % del importe neto asociado a pedidos entregados.

#### Hallazgos

La distribución comercial presenta concentración monetaria, aunque no reproduce estrictamente la proporción 80/20 utilizada habitualmente como referencia.

El resultado indica que se necesita una proporción relativamente amplia de la cartera para alcanzar el 80 % del importe neto.

#### Interpretación

La concentración constituye un indicador relevante para evaluar dependencia comercial, pero no demuestra por sí sola la existencia de un riesgo crítico.

Para profundizar en este comportamiento se desarrollará una clasificación ABC que permitirá segmentar la cartera según su contribución acumulada.

**Estado del análisis:** conforme.


In [166]:
%%sql

WITH resumen_clientes AS (
SELECT
    c.cliente_id,
    c.razon_social,
    c.segmento,
    COALESCE(SUM(t.importe_neto_linea_clp), 0) AS importe_neto_entregado_clp
FROM dim_cliente AS c
LEFT JOIN fact_detalle_pedidos AS t
    ON c.cliente_id = t.cliente_id
    AND t.estado = 'entregado'
GROUP BY c.cliente_id, c.razon_social, c.segmento
), concentracion_clientes AS (
SELECT
  *,
  ROW_NUMBER() OVER ( 
    ORDER BY importe_neto_entregado_clp DESC, cliente_id ASC
        ) AS posicion_cliente,
  ROUND(100.0 * importe_neto_entregado_clp /
   SUM(importe_neto_entregado_clp) OVER(), 2) AS participacion_pct,
    100.0 * SUM(importe_neto_entregado_clp) OVER(
    ORDER BY importe_neto_entregado_clp DESC, cliente_id ASC
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
) / SUM(importe_neto_entregado_clp) OVER() AS participacion_acumulada_pct
FROM resumen_clientes
), clasificacion_clientes AS (
SELECT 
    *,
    CASE
        WHEN LAG(participacion_acumulada_pct, 1 , 0) OVER( ORDER BY posicion_cliente) < 80 THEN 'A'
        WHEN LAG(participacion_acumulada_pct, 1 , 0) OVER( ORDER BY posicion_cliente) >= 80 
            AND  LAG(participacion_acumulada_pct, 1 , 0) OVER( ORDER BY posicion_cliente) < 95 THEN 'B'
        ELSE 'C'
    END AS clase_abc
FROM concentracion_clientes
)
SELECT 
    clase_abc,
    COUNT(*) AS total_clientes,
    ROUND(
        100.0 * COUNT(*) /
         (SELECT COUNT(*) FROM concentracion_clientes)
         , 2) AS pct_cartera,
    SUM(importe_neto_entregado_clp) AS importe_neto_entregado_clp,
    ROUND(
        100.0 * SUM(importe_neto_entregado_clp) / 
        (SELECT SUM(importe_neto_entregado_clp) FROM concentracion_clientes) 
        , 2) AS pct_importe_neto
FROM clasificacion_clientes
GROUP BY clase_abc
ORDER BY clase_abc ASC;

Running query in 'sqlite:///../data/cordillera_analytics.sqlite'

clase_abc,total_clientes,pct_cartera,importe_neto_entregado_clp,pct_importe_neto
A,212,35.33,3548826898,80.03
B,179,29.83,663931552,14.97
C,209,34.83,221443584,4.99



### 4.1.3 Clasificación ABC de clientes

#### Objetivo

Clasificar la cartera comercial en grupos A, B y C según su contribución acumulada al importe neto asociado a pedidos entregados.

#### Metodología

Se ordenaron los 600 clientes por importe neto entregado, de mayor a menor.

Mediante funciones de ventana se calcularon sus participaciones individuales y acumuladas.

La clasificación ABC se estableció según los siguientes criterios:

- **Clase A:** clientes necesarios para alcanzar al menos el 80 % del importe neto.
- **Clase B:** clientes siguientes, hasta alcanzar al menos el 95 %.
- **Clase C:** clientes restantes.

Se incluyó en cada clase al cliente que permite alcanzar su respectivo umbral.

#### Resultados

| Clase | Clientes | % cartera | % importe neto |
|---|---:|---:|---:|
| A | 212 | 35,33 % | 80,03 % |
| B | 179 | 29,83 % | 14,97 % |
| C | 209 | 34,83 % | 4,99 % |
| **Total** | **600** | **100 %** | **100 %** |

*Las pequeñas diferencias respecto del 100 % se deben al redondeo.*

#### Hallazgos e interpretación

La clase A concentra aproximadamente el 80 % del importe neto mediante el 35,33 % de los clientes registrados.

Por su parte, la clase C representa el 34,83 % de la cartera, pero aporta aproximadamente el 5 % del importe neto.

La distribución observada muestra una concentración comercial relevante, aunque menos pronunciada que una distribución estricta 80/20.

La clasificación constituye una herramienta para identificar clientes de mayor contribución monetaria y orientar análisis posteriores de dependencia comercial.

#### Limitaciones

La clasificación ABC se basa exclusivamente en importes netos asociados a pedidos entregados.

No considera directamente rentabilidad, comportamiento de pago, recurrencia reciente ni potencial de crecimiento.

Por tanto, no debe utilizarse como único criterio para definir prioridades o decisiones sobre la cartera comercial.


In [167]:
%%sql

WITH resumen_clientes AS (
SELECT
    c.cliente_id,
    c.razon_social,
    c.segmento,
    COALESCE(SUM(t.importe_neto_linea_clp), 0) AS importe_neto_entregado_clp
FROM dim_cliente AS c
LEFT JOIN fact_detalle_pedidos AS t
    ON c.cliente_id = t.cliente_id
    AND t.estado = 'entregado'
GROUP BY c.cliente_id, c.razon_social, c.segmento
), concentracion_clientes AS (
SELECT
  *,
  ROW_NUMBER() OVER ( 
    ORDER BY importe_neto_entregado_clp DESC, cliente_id ASC
        ) AS posicion_cliente,
  ROUND(100.0 * importe_neto_entregado_clp /
   SUM(importe_neto_entregado_clp) OVER(), 2) AS participacion_pct,
    100.0 * SUM(importe_neto_entregado_clp) OVER(
    ORDER BY importe_neto_entregado_clp DESC, cliente_id ASC
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
) / SUM(importe_neto_entregado_clp) OVER() AS participacion_acumulada_pct
FROM resumen_clientes
), clasificacion_clientes AS (
SELECT 
    *,
    CASE
        WHEN LAG(participacion_acumulada_pct, 1 , 0) OVER( ORDER BY posicion_cliente) < 80 THEN 'A'
        WHEN LAG(participacion_acumulada_pct, 1 , 0) OVER( ORDER BY posicion_cliente) >= 80 
            AND  LAG(participacion_acumulada_pct, 1 , 0) OVER( ORDER BY posicion_cliente) < 95 THEN 'B'
        ELSE 'C'
    END AS clase_abc
FROM concentracion_clientes
)
SELECT
    segmento,
    clase_abc,
    COUNT(*) AS total_clientes,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY segmento),2) AS pct_clientes_segmento,
    SUM(importe_neto_entregado_clp) AS importe_neto_entregado_clp
FROM clasificacion_clientes
GROUP BY segmento, clase_abc
ORDER BY segmento ASC, clase_abc ASC

Running query in 'sqlite:///../data/cordillera_analytics.sqlite'

segmento,clase_abc,total_clientes,pct_clientes_segmento,importe_neto_entregado_clp
Almacen,A,122,36.86,2125436382
Almacen,B,110,33.23,422579226
Almacen,C,99,29.91,105863988
Horeca,A,40,36.36,517492588
Horeca,B,14,12.73,39832432
Horeca,C,56,50.91,68354532
Minimarket,A,50,31.45,905897928
Minimarket,B,55,34.59,201519894
Minimarket,C,54,33.96,47225064



### 4.1.4 Composición ABC por segmento comercial

#### Objetivo

Evaluar la distribución de las clases ABC globales dentro de cada segmento comercial, identificando diferencias en la composición de la cartera.

#### Metodología

Se reutilizó la clasificación ABC de clientes previamente desarrollada.

Los clientes se agruparon por segmento y clase, calculando la cantidad de registros, la participación dentro de cada segmento y el importe neto asociado.

#### Resultados

| Segmento | Clase A | Clase B | Clase C |
|---|---:|---:|---:|
| Almacen | 36,86 % | 33,23 % | 29,91 % |
| Horeca | 36,36 % | 12,73 % | 50,91 % |
| Minimarket | 31,45 % | 34,59 % | 33,96 % |

#### Hallazgos

- Almacen presenta una distribución relativamente equilibrada entre las tres clases.
- Horeca concentra el 50,91 % de sus clientes en la clase C, la mayor proporción entre los segmentos.
- Minimarket presenta una participación de clientes A del 31,45 %, inferior a la observada en Almacen y Horeca.
- Las diferencias entre segmentos permiten identificar perfiles distintos de contribución monetaria.

#### Interpretación

La composición ABC evidencia diferencias en la distribución de clientes según su contribución al importe neto total de la empresa.

La mayor proporción de clientes C en Horeca constituye un aspecto que amerita investigación comercial adicional.

La clasificación fue calculada globalmente y no representa una clasificación ABC independiente para cada segmento.

### 4.1.5 Conclusión — Concentración comercial de clientes

El análisis permitió identificar que 212 clientes, equivalentes al 35,33 % de la cartera registrada, concentran el 80,03 % del importe neto asociado a pedidos entregados.

La distribución no sigue estrictamente la proporción 80/20, aunque presenta una concentración monetaria relevante.

La clasificación ABC permitió distinguir grupos de contribución y analizar su composición entre segmentos comerciales.

Estos hallazgos proporcionan una base para investigar dependencia comercial, recurrencia y rentabilidad individual.

**Limitación:** las clases ABC se construyeron exclusivamente a partir del importe neto histórico. No representan directamente rentabilidad, riesgo crediticio ni potencial comercial.

**Estado del análisis:** finalizado.



## 4.2 Concentración comercial de productos

### 4.2.1 Participación y concentración del importe neto

#### Objetivo

Evaluar cómo se distribuye el importe neto entregado entre los productos del catálogo, identificando su contribución individual y acumulada.

#### Metodología

Se utilizarán las vistas `dim_producto` y `fact_detalle_pedidos`.

Se conservarán los 180 productos registrados, incluidos aquellos sin pedidos entregados.

Los productos se ordenarán según su importe neto, de mayor a menor, para calcular la participación acumulada e identificar cuántos son necesarios para alcanzar el 80 % del total.

#### Consideraciones

La concentración monetaria no equivale directamente a concentración de rentabilidad.

Los productos sin entregas registradas permanecerán en el análisis con importe neto cero.

La clasificación por categorías corresponde a la información disponible en el modelo dimensional.


In [168]:
%%sql

WITH resumen_productos AS (
    SELECT
        p.producto_id,
        p.nombre_producto,
        p.nombre_categoria,
        SUM(t.importe_neto_linea_clp) AS importe_neto_entregado_clp
    FROM dim_producto AS p
    LEFT JOIN fact_detalle_pedidos AS t 
        ON t.producto_id = p.producto_id
        AND t.estado = 'entregado'
    GROUP BY p.producto_id, p.nombre_producto, p.nombre_categoria
), concentracion_productos AS (
    SELECT
        *,
        ROW_NUMBER () OVER(
            ORDER BY importe_neto_entregado_clp DESC, producto_id ASC
        ) AS posicion_producto,
        100.0 * SUM(importe_neto_entregado_clp) OVER(
            ORDER BY importe_neto_entregado_clp DESC, producto_id ASC
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) /
            SUM(importe_neto_entregado_clp) OVER() AS participacion_acumulada_pct
    FROM resumen_productos
    )
SELECT
    MIN(posicion_producto) AS productos_para_80_pct,
    ROUND(100.0 * MIN(posicion_producto) / (SELECT COUNT(*) FROM resumen_productos), 2) AS pct_productos_catalogo,
    ROUND(MIN(participacion_acumulada_pct), 2) AS pct_importe_alcanzado
FROM concentracion_productos
WHERE participacion_acumulada_pct >= 80

Running query in 'sqlite:///../data/cordillera_analytics.sqlite'

productos_para_80_pct,pct_productos_catalogo,pct_importe_alcanzado
94,52.22,80.05



#### Resultados y hallazgos

El análisis identificó que se requieren 94 de los 180 productos registrados, equivalentes al 52,22 % del catálogo, para alcanzar el 80,05 % del importe neto asociado a pedidos entregados.

#### Comparación con la concentración de clientes

| Indicador | Clientes | Productos |
|---|---:|---:|
| Registros analizados | 600 | 180 |
| Registros necesarios para alcanzar el 80 % | 212 | 94 |
| Porcentaje del total | 35,33 % | 52,22 % |
| Importe acumulado | 80,03 % | 80,05 % |

#### Interpretación

El importe neto presenta una mayor concentración relativa entre clientes que entre productos.

Mientras el 35,33 % de la cartera de clientes representa aproximadamente el 80 % del importe neto entregado, se necesita el 52,22 % del catálogo para alcanzar una participación equivalente.

Esto sugiere una distribución monetaria más amplia entre los productos comercializados.

#### Limitaciones

La concentración se calcula sobre importes netos históricos de pedidos entregados.

No permite evaluar directamente rentabilidad, dependencia operacional, disponibilidad de inventario ni riesgo comercial.

**Estado del análisis:** conforme.


In [169]:
%%sql

WITH resumen_productos AS (
    SELECT
        p.producto_id,
        p.nombre_producto,
        p.nombre_categoria,
        COALESCE(SUM(t.importe_neto_linea_clp), 0) AS importe_neto_entregado_clp
    FROM dim_producto AS p
    LEFT JOIN fact_detalle_pedidos AS t 
        ON t.producto_id = p.producto_id
        AND t.estado = 'entregado'
    GROUP BY p.producto_id, p.nombre_producto, p.nombre_categoria
), concentracion_productos AS (
    SELECT
        *,
        ROW_NUMBER () OVER(
            ORDER BY importe_neto_entregado_clp DESC, producto_id ASC
        ) AS posicion_producto,
        100.0 * SUM(importe_neto_entregado_clp) OVER(
            ORDER BY importe_neto_entregado_clp DESC, producto_id ASC
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) /
            SUM(importe_neto_entregado_clp) OVER() AS participacion_acumulada_pct
    FROM resumen_productos
    )
SELECT
    nombre_categoria,
    COUNT(*) AS productos_catalogo,
    SUM(CASE
        WHEN posicion_producto <= (SELECT MIN(posicion_producto) FROM concentracion_productos WHERE participacion_acumulada_pct>=80) THEN 1
        ELSE 0
    END) AS productos_top_80,
    SUM(CASE 
        WHEN posicion_producto <= (SELECT MIN(posicion_producto) FROM concentracion_productos WHERE participacion_acumulada_pct>=80)  THEN importe_neto_entregado_clp
        ELSE 0
        END) AS importe_top_80_clp
FROM concentracion_productos
GROUP BY nombre_categoria
ORDER BY importe_top_80_clp DESC;


Running query in 'sqlite:///../data/cordillera_analytics.sqlite'

nombre_categoria,productos_catalogo,productos_top_80,importe_top_80_clp
Abarrotes,30,25,1609454894
Bebidas,30,18,601113730
Snacks,30,14,382756732
Limpieza,30,14,363809748
Desechables,30,13,324012786
Cafeteria,30,10,268388588



### 4.2.2 Composición de productos de mayor contribución por categoría

#### Objetivo

Identificar la distribución por categoría de los productos necesarios para alcanzar al menos el 80 % del importe neto asociado a pedidos entregados.

#### Metodología

Se reutilizó el cálculo de participación acumulada por producto, identificando dinámicamente la posición en que se alcanza el umbral del 80 %.

Mediante agregaciones condicionales se calcularon los productos incluidos en ese conjunto y sus importes netos por categoría, conservando los 180 productos del catálogo.

#### Resultados

| Categoría | Productos incluidos | Importe neto (CLP) |
|---|---:|---:|
| Abarrotes | 25 | 1.609.454.894 |
| Bebidas | 18 | 601.113.730 |
| Snacks | 14 | 382.756.732 |
| Limpieza | 14 | 363.809.748 |
| Desechables | 13 | 324.012.786 |
| Cafetería | 10 | 268.388.588 |
| **Total** | **94** | **3.549.536.478** |

#### Hallazgos

- Se requieren 94 de los 180 productos registrados para alcanzar aproximadamente el 80 % del importe neto entregado.
- Abarrotes presenta la mayor representación, con 25 productos incluidos en el conjunto.
- Los productos de Abarrotes aportan el 45,34 % del importe neto correspondiente a los 94 productos seleccionados.
- Cafetería presenta la menor cantidad de productos incluidos, con 10 registros.

#### Interpretación comercial

El importe neto se distribuye entre todas las categorías, aunque Abarrotes destaca por su participación monetaria dentro del conjunto de mayor contribución.

La concentración por productos es relativamente menor que la observada entre clientes.

#### Conclusión de la sección 4.2

El análisis permitió cuantificar la concentración del importe neto a nivel de producto y examinar la composición por categoría del conjunto de mayor contribución.

Los resultados constituyen una base para investigar diferencias de rentabilidad, desempeño del catálogo y oportunidades de optimización comercial.

**Limitación:** la clasificación considera contribución monetaria histórica, no rentabilidad neta, rotación de inventario ni demanda futura.

**Estado del análisis:** finalizado.



## 4.3 Rentabilidad bruta estimada

### 4.3.1 Comparación de márgenes por cliente

#### Objetivo

Comparar el margen bruto estimado de los clientes con el margen global de la distribuidora, identificando diferencias entre contribución monetaria y rentabilidad porcentual.

#### Metodología

Se consolidaron los importes netos, costos asociados y márgenes brutos estimados de los pedidos entregados a nivel de cliente.

El margen porcentual se calculó como la relación entre el margen bruto estimado y el importe neto entregado.

Se utilizó el margen global ponderado como referencia para calcular las diferencias individuales en puntos porcentuales.

#### Resultados y hallazgos

El margen bruto estimado global corresponde al 28,46 %.

Entre los diez clientes de mayor importe neto:

- Siete presentan márgenes porcentuales superiores al promedio global.
- Tres presentan márgenes inferiores al promedio global.
- Los márgenes individuales oscilan entre 27,73 % y 28,89 %.
- El cliente Almacen 0057 registra la mayor brecha negativa del Top 10, con -0,73 puntos porcentuales.

#### Interpretación comercial

Los clientes de mayor contribución monetaria no necesariamente presentan los mayores márgenes porcentuales.

Las diferencias observadas entre los diez clientes principales son relativamente acotadas.

Un margen porcentual inferior al promedio no implica necesariamente una contribución monetaria insuficiente, por lo que ambos indicadores deben evaluarse conjuntamente.

#### Limitaciones

Los márgenes representan estimaciones basadas en importes y costos registrados en las líneas de pedido.

No incorporan gastos operacionales, financieros ni otros elementos necesarios para determinar la rentabilidad neta por cliente.


In [170]:
%%sql

WITH resumen_clientes AS (
SELECT
    c.cliente_id,
    c.razon_social,
    c.segmento,
    SUM(t.importe_neto_linea_clp) AS importe_neto_entregado_clp,
    SUM(t.costo_linea_clp) AS costo_entregado_clp,
    SUM(t.margen_bruto_estimado_linea_clp) AS margen_bruto_estimado_clp,
    ROUND(100.0 * SUM(t.margen_bruto_estimado_linea_clp) / SUM(t.importe_neto_linea_clp), 2) AS pct_margen_bruto_estimado

FROM dim_cliente AS c
INNER JOIN fact_detalle_pedidos AS t
    ON c.cliente_id = t.cliente_id
    AND t.estado = 'entregado'
GROUP BY c.cliente_id, c.razon_social, c.segmento
)
SELECT
    *,
    ROUND(
    100.0 * margen_bruto_estimado_clp
        / NULLIF(importe_neto_entregado_clp, 0)
    -
    100.0 * SUM(margen_bruto_estimado_clp) OVER()
        / NULLIF(SUM(importe_neto_entregado_clp) OVER(), 0),
    2
) AS brecha_margen_pp
FROM resumen_clientes
ORDER BY importe_neto_entregado_clp DESC
LIMIT 10;

Running query in 'sqlite:///../data/cordillera_analytics.sqlite'

cliente_id,razon_social,segmento,importe_neto_entregado_clp,costo_entregado_clp,margen_bruto_estimado_clp,pct_margen_bruto_estimado,brecha_margen_pp
52,Almacen Ficticio Cordillera 0052,Almacen,48125814,34374150,13751664,28.57,0.11
67,Almacen Ficticio Cordillera 0067,Almacen,45871116,32713086,13158030,28.68,0.22
57,Almacen Ficticio Cordillera 0057,Almacen,45676530,33008850,12667680,27.73,-0.73
48,Minimarket Ficticio Cordillera 0048,Minimarket,45091854,32078202,13013652,28.86,0.4
54,Minimarket Ficticio Cordillera 0054,Minimarket,43859664,31400874,12458790,28.41,-0.06
63,Almacen Ficticio Cordillera 0063,Almacen,43667496,31442856,12224640,27.99,-0.47
74,Almacen Ficticio Cordillera 0074,Almacen,43276812,30775806,12501006,28.89,0.42
73,Almacen Ficticio Cordillera 0073,Almacen,42543000,30299520,12243480,28.78,0.32
72,Almacen Ficticio Cordillera 0072,Almacen,42428298,30339054,12089244,28.49,0.03
64,Almacen Ficticio Cordillera 0064,Almacen,42312942,30143778,12169164,28.76,0.3


In [171]:

%%sql

WITH resumen_clientes AS (
    SELECT
        c.cliente_id,
        COALESCE(SUM(t.importe_neto_linea_clp), 0)
            AS importe_neto_entregado_clp,
        COALESCE(SUM(t.margen_bruto_estimado_linea_clp), 0)
            AS margen_bruto_estimado_clp
    FROM dim_cliente AS c
    LEFT JOIN fact_detalle_pedidos AS t
        ON c.cliente_id = t.cliente_id
        AND t.estado = 'entregado'
    GROUP BY c.cliente_id
),

concentracion_clientes AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            ORDER BY importe_neto_entregado_clp DESC, cliente_id ASC
        ) AS posicion_cliente,
        100.0 * SUM(importe_neto_entregado_clp) OVER (
            ORDER BY importe_neto_entregado_clp DESC, cliente_id ASC
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) / NULLIF(SUM(importe_neto_entregado_clp) OVER (), 0)
            AS participacion_acumulada_pct
    FROM resumen_clientes
),

acumulado_anterior AS (
    SELECT
        *,
        LAG(participacion_acumulada_pct, 1, 0) OVER (
            ORDER BY posicion_cliente
        ) AS acumulado_anterior_pct
    FROM concentracion_clientes
),

clasificacion_clientes AS (
    SELECT
        *,
        CASE
            WHEN acumulado_anterior_pct < 80 THEN 'A'
            WHEN acumulado_anterior_pct < 95 THEN 'B'
            ELSE 'C'
        END AS clase_abc
    FROM acumulado_anterior
),

resumen_abc AS (
    SELECT
        clase_abc,
        COUNT(*) AS total_clientes,
        SUM(importe_neto_entregado_clp)
            AS importe_neto_entregado_clp,
        SUM(margen_bruto_estimado_clp)
            AS margen_bruto_estimado_clp
    FROM clasificacion_clientes
    GROUP BY clase_abc
)

SELECT
    clase_abc,
    total_clientes,
    importe_neto_entregado_clp,
    margen_bruto_estimado_clp,

    ROUND(
        100.0 * margen_bruto_estimado_clp /
        NULLIF(importe_neto_entregado_clp, 0),
        2
    ) AS pct_margen_bruto_estimado,

    ROUND(
        100.0 * importe_neto_entregado_clp /
        NULLIF(SUM(importe_neto_entregado_clp) OVER (), 0),
        2
    ) AS pct_importe_neto,

    ROUND(
        100.0 * margen_bruto_estimado_clp /
        NULLIF(SUM(margen_bruto_estimado_clp) OVER (), 0),
        2
    ) AS pct_margen_bruto

FROM resumen_abc
ORDER BY clase_abc ASC;


Running query in 'sqlite:///../data/cordillera_analytics.sqlite'

clase_abc,total_clientes,importe_neto_entregado_clp,margen_bruto_estimado_clp,pct_margen_bruto_estimado,pct_importe_neto,pct_margen_bruto
A,212,3548826898,1007542254,28.39,80.03,79.83
B,179,663931552,190223996,28.65,14.97,15.07
C,209,221443584,64336498,29.05,4.99,5.1



### 4.3.2 Contribución al margen bruto por clase ABC

#### Objetivo

Evaluar si la concentración del importe neto entre las clases ABC de clientes también se refleja en su contribución al margen bruto estimado.

#### Metodología

Se reutilizó la clasificación ABC global construida a partir del importe neto entregado.

Para cada clase se calcularon el importe neto, el margen bruto estimado, la tasa de margen y la participación relativa en ambas medidas monetarias.

#### Resultados

| Clase | Clientes | % importe neto | % margen bruto | Tasa de margen |
|---|---:|---:|---:|---:|
| A | 212 | 80,03 % | 79,83 % | 28,39 % |
| B | 179 | 14,97 % | 15,07 % | 28,65 % |
| C | 209 | 4,99 % | 5,10 % | 29,05 % |

#### Hallazgos

La clase A concentra la mayor parte del importe neto y del margen bruto estimado en términos absolutos.

Sin embargo, presenta la menor tasa porcentual de margen entre las tres clases, con un 28,39 %.

La clase C registra la mayor tasa de margen (29,05 %), aunque representa únicamente el 5,10 % del margen bruto estimado total.

#### Interpretación comercial

La concentración de importe neto y margen bruto presenta distribuciones similares, pero no idénticas.

Los clientes de mayor contribución monetaria no necesariamente registran las tasas de margen más elevadas.

Las diferencias entre clases son relativamente reducidas y no justifican, por sí solas, cambios en la estrategia comercial.

#### Conclusión

La clasificación ABC basada en importe neto permite identificar clientes de mayor contribución monetaria, pero no sustituye la evaluación de rentabilidad bruta estimada.

La priorización comercial debería considerar conjuntamente importe neto, margen absoluto, margen porcentual y otras variables operacionales.



### 4.3.3 Comparación de márgenes por producto y categoría

#### Objetivo

Identificar productos cuyo margen bruto estimado porcentual sea inferior al margen agregado de su categoría, priorizando aquellos con mayor importe neto entregado.

#### Metodología

Se calcularán importes netos y márgenes brutos estimados a nivel de producto.

Posteriormente se determinará el margen porcentual ponderado de cada categoría, utilizando los importes y márgenes agregados de sus productos.

La diferencia entre el margen individual y el margen de categoría se expresará en puntos porcentuales.

#### Consideraciones

Los resultados permitirán identificar productos que ameritan revisión comercial, pero no demostrarán por sí solos la existencia de precios incorrectos ni oportunidades de mejora efectivamente realizables.


In [172]:

%%sql

WITH resumen_productos AS (
    SELECT
        p.producto_id,
        p.nombre_producto,
        p.nombre_categoria,

        SUM(t.importe_neto_linea_clp)
            AS importe_neto_entregado_clp,

        SUM(t.margen_bruto_estimado_linea_clp)
            AS margen_bruto_estimado_clp

    FROM dim_producto AS p

    INNER JOIN fact_detalle_pedidos AS t
        ON p.producto_id = t.producto_id
        AND t.estado = 'entregado'

    GROUP BY
        p.producto_id,
        p.nombre_producto,
        p.nombre_categoria
),

comparacion_margenes AS (
    SELECT
        *,

        100.0 * margen_bruto_estimado_clp /
        NULLIF(importe_neto_entregado_clp, 0)
            AS pct_margen_producto,

        100.0 * SUM(margen_bruto_estimado_clp) OVER (
            PARTITION BY nombre_categoria
        ) /
        NULLIF(
            SUM(importe_neto_entregado_clp) OVER (
                PARTITION BY nombre_categoria
            ), 0
        ) AS pct_margen_categoria

    FROM resumen_productos
)

SELECT
    nombre_producto,
    nombre_categoria,
    importe_neto_entregado_clp,
    margen_bruto_estimado_clp,

    ROUND(pct_margen_producto, 2)
        AS pct_margen_producto,

    ROUND(pct_margen_categoria, 2)
        AS pct_margen_categoria,

    ROUND(
        pct_margen_producto - pct_margen_categoria,
        2
    ) AS brecha_margen_pp

FROM comparacion_margenes

WHERE importe_neto_entregado_clp > 0
    AND pct_margen_producto < pct_margen_categoria

ORDER BY
    importe_neto_entregado_clp DESC,
    producto_id ASC

LIMIT 10;


Running query in 'sqlite:///../data/cordillera_analytics.sqlite'

nombre_producto,nombre_categoria,importe_neto_entregado_clp,margen_bruto_estimado_clp,pct_margen_producto,pct_margen_categoria,brecha_margen_pp
Harina 1 kg Marca Ficticia 2,Abarrotes,109911434,24037894,21.87,29.44,-7.57
Arroz 1 kg Marca Ficticia 1,Abarrotes,94364198,19949806,21.14,29.44,-8.3
Harina 1 kg Marca Ficticia 1,Abarrotes,94029090,25470168,27.09,29.44,-2.36
Azucar 1 kg Marca Ficticia 2,Abarrotes,58927000,11606488,19.7,29.44,-9.75
Azucar 1 kg Marca Ficticia 4,Abarrotes,57596060,12949846,22.48,29.44,-6.96
Aceite 1 L Marca Ficticia 4,Abarrotes,39744500,8691352,21.87,29.44,-7.58
Bebida isotónica 500 ml Marca Ficticia 1,Bebidas,37317528,8221498,22.03,28.86,-6.82
Bebida 1.5 L Marca Ficticia 1,Bebidas,34305520,7563438,22.05,28.86,-6.81
Aceite 1 L Marca Ficticia 5,Abarrotes,34222336,9154768,26.75,29.44,-2.69
Arroz 1 kg Marca Ficticia 3,Abarrotes,33913542,7085898,20.89,29.44,-8.55



#### Resultados — Comparación de márgenes por producto y categoría

Se identificaron los diez productos de mayor importe neto cuyo margen bruto estimado porcentual es inferior al margen ponderado de su respectiva categoría.

De los diez productos seleccionados, ocho pertenecen a Abarrotes y dos a Bebidas.

En conjunto, representan $594.331.208 CLP de importe neto entregado y $134.731.156 CLP de margen bruto estimado, equivalentes a una tasa agregada de margen del 22,67 %.

#### Hallazgos

- Harina 1 kg Marca Ficticia 2 presenta el mayor importe neto entre los productos seleccionados, con $109.911.434 CLP y una brecha de -7,57 puntos porcentuales.
- Arroz 1 kg Marca Ficticia 1 presenta $94.364.198 CLP de importe neto y una brecha de -8,30 puntos porcentuales.
- Azucar 1 kg Marca Ficticia 2 registra la mayor brecha negativa del grupo, con -9,75 puntos porcentuales respecto de Abarrotes.
- Los diez productos representan aproximadamente el 13,40 % del importe neto general, pero solo el 10,67 % del margen bruto estimado total.

#### Interpretación comercial

Los resultados identifican productos relevantes por importe neto que presentan tasas de margen inferiores a las referencias ponderadas de sus categorías.

Estos casos constituyen candidatos para revisar precios, costos de adquisición, descuentos y composición de ventas.

Sin embargo, no puede concluirse que exista una política de precios incorrecta ni que sea posible incrementar sus márgenes sin afectar el volumen comercial.

---

## 4.4 Síntesis ejecutiva y conclusiones

### 4.4.1 Resumen del desempeño económico

Durante el período analizado, correspondiente a pedidos registrados entre 2024 y septiembre de 2026, la distribuidora presenta los siguientes indicadores asociados a pedidos entregados:

| Indicador | Resultado |
|---|---:|
| Pedidos entregados | 14.734 |
| Importe neto entregado | $4.434.202.034 |
| Costo asociado | $3.172.099.286 |
| Margen bruto estimado | $1.262.102.748 |
| Margen bruto estimado (%) | 28,46 % |

Estos indicadores corresponden al comportamiento histórico del conjunto de datos ficticio utilizado para el proyecto.

### 4.4.2 Principales hallazgos

**Concentración de clientes**

Un total de 212 clientes, equivalentes al 35,33 % de la cartera registrada, concentra el 80,03 % del importe neto entregado.

La concentración monetaria es más pronunciada entre clientes que entre productos.

**Concentración de productos**

Se requieren 94 de los 180 productos del catálogo para alcanzar aproximadamente el 80 % del importe neto entregado.

Abarrotes presenta la mayor representación dentro de este conjunto, con 25 productos.

**Rentabilidad por clase ABC**

La clase A concentra el 80,03 % del importe neto y el 79,83 % del margen bruto estimado.

Su tasa de margen es del 28,39 %, ligeramente inferior a las tasas observadas en las clases B y C.

Esto evidencia que una mayor contribución monetaria no implica necesariamente una mayor tasa de margen.

**Rentabilidad por producto**

Se identificaron productos de importe neto relevante con márgenes inferiores a los promedios ponderados de sus categorías.

Los diez casos seleccionados presentan una tasa conjunta de margen del 22,67 %, por debajo del margen bruto global estimado del 28,46 %.

### 4.4.3 Oportunidades de análisis y gestión

**1. Seguimiento de clientes de alta contribución**

Priorizar el monitoreo de los clientes de clase A, considerando su importancia dentro del importe neto y del margen bruto absoluto.

**2. Evaluación de márgenes por producto**

Revisar los productos de elevado importe neto y brechas negativas de margen, considerando precios, costos, descuentos y características comerciales.

**3. Seguimiento por categoría**

Incorporar indicadores que permitan observar conjuntamente importe neto, margen absoluto y margen porcentual por categoría.

**4. Monitoreo periódico de concentración**

Evaluar la evolución de las participaciones comerciales para detectar cambios en la dependencia relativa de clientes y productos.

### 4.4.4 Limitaciones

- Los datos corresponden a un conjunto ficticio y no representan operaciones reales.
- Los resultados se basan en pedidos entregados, no necesariamente facturados o cobrados.
- Los márgenes son estimaciones brutas y no consideran gastos operacionales, financieros ni tributarios.
- El análisis no incorpora inventario disponible, costos logísticos ni comportamiento de cobranza.
- Las diferencias de margen no permiten inferir causalidad ni determinar automáticamente decisiones óptimas de precios.

### 4.4.5 Conclusión general

El análisis permitió caracterizar la concentración comercial y la rentabilidad bruta estimada de la distribuidora desde las dimensiones de clientes, productos y categorías.

Los resultados muestran que la contribución monetaria y la rentabilidad porcentual son dimensiones complementarias que deben analizarse conjuntamente.

La identificación de clientes de alta contribución y productos con márgenes inferiores a sus categorías proporciona una base para desarrollar indicadores de seguimiento y orientar investigaciones comerciales posteriores.

El modelo analítico y los resultados documentados constituyen los fundamentos de la siguiente etapa del proyecto: construcción de un dashboard ejecutivo en Power BI.



### Estado final del análisis

El notebook consolida el análisis de concentración comercial y rentabilidad bruta estimada de Distribuidora Cordillera.

Se desarrollaron consultas SQL para evaluar:

- Concentración monetaria de clientes y productos.
- Clasificación ABC de la cartera comercial.
- Participación de clientes y productos en el importe neto.
- Contribución al margen bruto estimado.
- Diferencias de rentabilidad entre productos y categorías.
- Oportunidades de seguimiento comercial.

Los análisis utilizan el modelo dimensional construido previamente y consideran únicamente pedidos entregados para las medidas comerciales.

Las consultas y sus resultados proporcionan indicadores y evidencia para desarrollar una solución de Business Intelligence en Power BI.

**Estado:** análisis completado y notebook ejecutado correctamente.
